# 10 Years of FAIR Principles — Notebook Index & Runner

This notebook is the table of contents for the analysis notebooks in this folder, and a runner
that executes all of them in order against `../data/processed/pmc_fair.db`.

All notebooks share one database. Publication year (`article.year`) is a persisted, indexed
column — `pub_year` as the anchor, `epub_date` only overriding it when within ±1 year (see
`01_corpus_overview.ipynb` §0 for the full rationale). Every notebook's setup cell checks for this
column and runs the one-time migration itself if it's missing, so notebooks are safe to run in any
order — this runner just reproduces the intended reading order.

## Known issues

- **[RESOLVED] `02_das_categories.ipynb` vs `05_notebooks_in_das.ipynb` near-duplicates.**
  `02` originally had 26/27 code cells byte-identical to `05` (both analysed Jupyter-notebook
  sharing in DAS text, under different filenames). `02_das_categories.ipynb` has since been
  rewritten to its own distinct topic — DAS category structure, quality, and trajectories — and
  no longer overlaps with `05`. Left here as a record in case the old duplicate content
  resurfaces from a stale copy.
- **[RESOLVED] `01_corpus_overview.ipynb` vs `11_geographical.ipynb` figure/export collisions.**
  `11` rebuilds the same corpus-overview baseline as `01` before extending into its own
  discipline/OA/funder/geographic sections, and had inherited `01`'s exact figure filenames
  (`fig1_corpus_overview.png` … `fig11_before_after.png`) and export directory
  (`fair_analysis_exports/`) — both bare-numbered, unlike every other notebook's
  `fig_<shortcode>_NN_description.png` / `fair_<topic>_exports/` convention. Running both (as
  this index does) meant the second to run silently overwrote the first one's output. Fixed by
  giving `01` the `fig_corpus_*` / `fair_corpus_exports/` namespace and `11` the `fig_geo_*` /
  `fair_geographical_exports/` namespace; also normalised `02`'s export directory to
  `fair_das_categories_exports/` for the same convention. Full old→new mapping in
  `changed_files.txt`. All notebooks were audited for figure/export-name collisions and
  meaningfulness; no other notebook had either problem.
- **[RESOLVED] Two unused notebooks removed, and the folder renumbered consecutively.**
  `07_gender_analysis_old.ipynb` (an early draft superseded by `07_gender_analysis.ipynb`, as it
  was then numbered) and `12_link_rot.ipynb` (link-rot/URL-survival analysis -- its output fed no
  figure or statistic actually reported in the manuscript, and the Methods section's own
  description of the pipeline already noted Stage 8 link-rot checking as outside the analyses
  reported there) were both dropped. The remaining 13 topic notebooks were then renumbered to run
  `00`-`12` with no gaps, replacing the old scheme in which `01`-`15` skipped `03` and `13`
  (reserved to avoid confusion with `pipeline/`'s own `01`-`08`-numbered stage scripts, a
  precaution no longer needed since no notebook and no pipeline script now share a number) and
  interleaved the two since-removed notebooks. No figure, export, or database table name is
  affected: those are all named by topic shortcode (`fig_corpus_*`, `fig_mandate_*`, ...) rather
  than by notebook number. Every in-notebook and in-manuscript filename reference was updated to
  match at the same time.

## Notebooks, in run order

| # | Notebook | Covers |
|---|----------|--------|
| 00 | `00_database_overview.ipynb` | Database sanity check: row counts, schema spot-checks, year coverage |
| 01 | `01_corpus_overview.ipynb` | Corpus-wide baseline: article counts, DAS categories, data-sharing score trend, repository landscape, accession numbers, secondary quality signals, F-UJI results, before/after statistics |
| 02 | `02_das_categories.ipynb` | DAS category structure deep-dive: composition, text richness, secondary signals, repository infrastructure tier, restrictive-sharing trend, publisher/journal trajectories |
| 03 | `03_jats4r_compliance.ipynb` | JATS4R XML-tagging compliance for Data Availability Statements |
| 04 | `04_ethics_statements.ipynb` | Ethics statements vs. DAS: co-occurrence, JATS4R ethics-tagging compliance |
| 05 | `05_notebooks_in_das.ipynb` | Jupyter notebook sharing detected in DAS text: prevalence, platforms, quality signals, adoption over time |
| 06 | `06_gender_analysis.ipynb` | Author gender (inferred from given names) vs. FAIR data-sharing behaviour |
| 07 | `07_diamond_oa.ipynb` | Diamond / Gold / Hybrid / Subscription open-access status vs. FAIR data sharing |
| 08 | `08_mandate_compliance.ipynb` | Funder and journal mandate compatibility, compliance tiers over time |
| 09 | `09_reproducibility.ipynb` | Computational reproducibility vs. FAIR data sharing |
| 10 | `10_duc_oami.ipynb` | Digital Use Conditions, Indigenous Data Sovereignty, and OAMI (Open Access to Material and Information) |
| 11 | `11_geographical.ipynb` | Cross-cutting breakdowns (discipline, OA, funder, author, MeSH) culminating in geographical/country-level FAIR analysis |
| 12 | `12_journal_profiles.ipynb` | Journal-level integrated profile: FAIR, funder, notebook-sharing, and author/gender view |

(This file, `index.ipynb`, is the only notebook in the folder without a number -- it is the table
of contents and runner, not a topic analysis in its own right.)

## Output locations

Figures and CSV exports live outside `notebooks/`, as siblings to it: `../figures/` for every
`.png` and `../results/` for every export directory. Each notebook saves figures as
`../figures/fig_<shortcode>_NN_description.png` and exports CSVs to
`../results/fair_<topic>_exports/` (e.g. `../figures/fig_mandate_07_publishers.png`,
`../results/fair_mandate_exports/`) — a notebook-specific namespace within each shared folder,
so no two notebooks can ever write into the same file, and `notebooks/` itself stays free of
generated output. Every notebook creates both directories itself (`Path(...).mkdir(parents=True,
exist_ok=True)`) before writing, so nothing needs to exist ahead of time. If you add a new
notebook or a new figure, follow this same convention rather than a bare `fig1.png`/`fig2.png`
style saved into `notebooks/` itself — see "Known issues" above for what goes wrong otherwise.

## What the runner does

For each notebook in order: run `jupyter nbconvert --to notebook --execute --inplace`, capture
success/failure and wall-clock time, and continue to the next notebook even if one fails (all
failures are reported together at the end rather than stopping the whole run).

In [ ]:
import subprocess, sys, time
from pathlib import Path

NOTEBOOKS = [
    ("00_database_overview.ipynb", "Database sanity check"),
    ("01_corpus_overview.ipynb",   "Corpus overview & baseline trends"),
    ("02_das_categories.ipynb",    "DAS category structure deep-dive"),
    ("03_jats4r_compliance.ipynb", "JATS4R XML-tagging compliance"),
    ("04_ethics_statements.ipynb", "Ethics statements vs DAS"),
    ("05_notebooks_in_das.ipynb",  "Jupyter notebooks in DAS"),
    ("06_gender_analysis.ipynb",   "Author gender vs FAIR data-sharing behaviour"),
    ("07_diamond_oa.ipynb",        "Diamond/Gold/Hybrid OA vs FAIR sharing"),
    ("08_mandate_compliance.ipynb","Funder/journal mandate compliance"),
    ("09_reproducibility.ipynb",   "Computational reproducibility vs FAIR sharing"),
    ("10_duc_oami.ipynb",          "Digital Use Conditions, Indigenous Data Sovereignty, OAMI"),
    ("11_geographical.ipynb",      "Cross-cutting + geographical FAIR analysis"),
    ("12_journal_profiles.ipynb", "Journal-level integrated FAIR profile"),
]

NOTEBOOK_DIR = Path(".")
TIMEOUT_S = 1800  # per-notebook execution timeout passed to nbconvert's own ExecutePreprocessor


In [ ]:
results = []

for fname, desc in NOTEBOOKS:
    path = NOTEBOOK_DIR / fname
    print(f"\n{'='*70}\nRunning {fname}  --  {desc}\n{'='*70}")
    if not path.exists():
        print(f"  SKIPPED -- file not found: {path}")
        results.append((fname, "missing", 0.0, ""))
        continue

    t0 = time.time()
    proc = subprocess.run(
        [sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
         "--inplace", str(fname), f"--ExecutePreprocessor.timeout={TIMEOUT_S}"],
        cwd=NOTEBOOK_DIR, capture_output=True, text=True,
    )
    elapsed = time.time() - t0

    if proc.returncode == 0:
        print(f"  OK in {elapsed:.1f}s")
        results.append((fname, "ok", elapsed, ""))
    else:
        # keep only the last part of stderr -- nbconvert error traces can be long
        err_tail = proc.stderr.strip().splitlines()
        err_tail = "\n".join(err_tail[-15:])
        print(f"  FAILED after {elapsed:.1f}s\n{err_tail}")
        results.append((fname, "FAILED", elapsed, err_tail))

In [ ]:
print(f"\n{'='*70}\nSUMMARY\n{'='*70}")
total = 0.0
for fname, status, elapsed, _ in results:
    total += elapsed
    mark = "✅" if status == "ok" else ("⚠️ " if status == "missing" else "❌")
    print(f"  {mark} {fname:<32} {status:<8} {elapsed:>7.1f}s")
print(f"\nTotal wall-clock time: {total:.1f}s ({total/60:.1f} min)")

failed = [r for r in results if r[1] == "FAILED"]
if failed:
    print(f"\n{len(failed)} notebook(s) FAILED -- see error output above for each.")
else:
    print("\nAll notebooks executed successfully.")